<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/08.%20Tree-Based%20Algorithms%20and%20Ensemble%20Methods/ch08_tree_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 8: Tree-Based Algorithms and Ensemble Methods

## Decision Trees: Pemisahan Rekursif

Decision tree mempartisi ruang fitur secara rekursif menggunakan aturan if-then. Setiap pemisahan dipilih untuk memaksimalkan **purity** (kemurnian) node anak.

### Kriteria Pemisahan

**Gini Impurity** (default di scikit-learn):
$$I_G(t) = 1 - \sum_{k=1}^{K} p_k^2$$

**Entropy (Information Gain)**:
$$H(t) = -\sum_{k=1}^{K} p_k \log_2 p_k$$

**Information Gain** dari split:
$$\Delta H = H(\text{parent}) - \left[\frac{n_L}{n} H(L) + \frac{n_R}{n} H(R)\right]$$

## Ensemble Methods

### Bagging (Bootstrap Aggregating)

Bagging melatih $T$ model pada subsets bootstrap dari data:
$$\hat{y} = \frac{1}{T} \sum_{t=1}^{T} f_t(\mathbf{x})$$

Random Forest = Bagging + Random Feature Subspace ($\sqrt{p}$ fitur per split).

### Boosting

Boosting melatih model secara **sekuensial**, setiap model baru berfokus pada kesalahan model sebelumnya:
$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x})$$

di mana $h_m$ adalah base learner yang meminimalkan residual $F_{m-1}$.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cancer, y_cancer, test_size=0.25, random_state=42, stratify=y_cancer
)
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')


## 8.1 Decision Tree: Depth dan Pruning

Depth pohon mengontrol kompleksitas model:
- **max_depth kecil** → underfitting (high bias)
- **max_depth besar** → overfitting (high variance)


In [ ]:
# Effect of max_depth on performance
depths = [1, 2, 3, 5, 7, 10, None]
train_acc, test_acc, cv_acc = [], [], []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_tr, y_tr)
    train_acc.append(dt.score(X_tr, y_tr))
    test_acc.append(dt.score(X_te, y_te))
    cv = cross_val_score(dt, X_cancer, y_cancer, cv=5).mean()
    cv_acc.append(cv)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
depth_labels = [str(d) if d else 'None' for d in depths]

axes[0].plot(depth_labels, train_acc, 'b-o', label='Train')
axes[0].plot(depth_labels, test_acc, 'r-s', label='Test')
axes[0].plot(depth_labels, cv_acc, 'g-^', label='5-CV')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Decision Tree: Depth vs Performance')
axes[0].legend()

# Visualize tree with max_depth=3
best_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
best_dt.fit(X_tr, y_tr)

from sklearn.tree import export_text
axes[1].axis('off')
rules = export_text(best_dt, feature_names=list(cancer.feature_names), max_depth=2)
axes[1].text(0.01, 0.99, rules[:400], transform=axes[1].transAxes,
    fontsize=6, va='top', fontfamily='monospace')
axes[1].set_title('Decision Tree Rules (depth=3, truncated)')

plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(cv_acc)]
print(f'Best depth by CV: {best_depth}, CV acc: {max(cv_acc):.4f}')


## 8.2 Feature Importance dari Decision Tree

Feature importance diukur dari total **impurity decrease** yang dikontribusikan oleh setiap fitur, berbobot dengan jumlah sampel yang melalui node tersebut:

$$\text{Importance}(f) = \sum_{t \in \text{nodes where } f \text{ is used}} \frac{n_t}{n} \cdot \Delta I_G(t)$$


In [ ]:
# Feature importance from Decision Tree
dt_full = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_full.fit(X_tr, y_tr)

importances = dt_full.feature_importances_
feature_names = cancer.feature_names

# Sort by importance
sorted_idx = np.argsort(importances)[::-1][:15]  # Top 15

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(sorted_idx)))
ax.barh([feature_names[i] for i in sorted_idx[::-1]],
        importances[sorted_idx[::-1]], color=colors, alpha=0.8)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 15 Feature Importances - Decision Tree (Breast Cancer)')
for i, (feat_idx, imp) in enumerate(zip(sorted_idx[::-1], importances[sorted_idx[::-1]])):
    ax.text(imp + 0.002, i, f'{imp:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()


## 8.3 Random Forest

Random Forest mengurangi variance decision tree melalui dua sumber randomness:
1. **Bootstrap sampling**: setiap tree dilatih pada sample acak dengan pengembalian
2. **Random feature subset**: setiap split hanya mempertimbangkan $\sqrt{p}$ fitur acak

Variance model ensemble turun sebagai:
$$\text{Var}\left(\frac{1}{T}\sum_{t=1}^T f_t\right) = \rho \sigma^2 + \frac{1-\rho}{T}\sigma^2$$

di mana $\rho$ adalah korelasi antar trees. Semakin kecil $\rho$ (lebih random), semakin kecil variance ensemble.


In [ ]:
# Random Forest: n_estimators effect
n_est_range = [5, 10, 25, 50, 100, 200, 500]
rf_cv_accs = []

for n in n_est_range:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    cv = cross_val_score(rf, X_cancer, y_cancer, cv=5).mean()
    rf_cv_accs.append(cv)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(n_est_range, rf_cv_accs, 'g-o', lw=2)
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('5-Fold CV Accuracy')
axes[0].set_title('Random Forest: n_estimators vs CV Accuracy')
axes[0].axhline(max(rf_cv_accs), color='red', linestyle='--', alpha=0.5, label=f'Max: {max(rf_cv_accs):.4f}')
axes[0].legend()

# Feature importance comparison: DT vs RF
rf_best = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_best.fit(X_tr, y_tr)
rf_importances = rf_best.feature_importances_
sorted_rf = np.argsort(rf_importances)[::-1][:10]

width = 0.35
x = np.arange(10)
axes[1].bar(x - width/2, importances[sorted_rf], width, label='DT', alpha=0.7, color='steelblue')
axes[1].bar(x + width/2, rf_importances[sorted_rf], width, label='RF', alpha=0.7, color='tomato')
axes[1].set_xticks(x)
axes[1].set_xticklabels([feature_names[i][:12] for i in sorted_rf], rotation=45, ha='right', fontsize=8)
axes[1].set_title('Feature Importance: DT vs RF (Top 10)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'RF Test Accuracy: {rf_best.score(X_te, y_te):.4f}')


## 8.4 Gradient Boosting

Gradient Boosting membangun model secara aditif dengan meminimalkan fungsi loss menggunakan gradient descent di function space:

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x})$$

di mana $h_m$ difit untuk memprediksi **pseudo-residuals** (negative gradient of loss):

$$r_{im} = -\left[\frac{\partial L(y_i, F(\mathbf{x}_i))}{\partial F(\mathbf{x}_i)}\right]_{F=F_{m-1}}$$


In [ ]:
# Compare ensemble methods
models = {
    'Decision Tree (d=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Bagging (DT base)': BaggingClassifier(DecisionTreeClassifier(max_depth=5), n_estimators=50, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
}

print('=== Ensemble Methods Comparison (Breast Cancer) ===')
print(f'{"Model":<25} {"Test Acc":>10} {"5-CV Acc":>10}')
print('-' * 47)
results_ensemble = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    test_acc = model.score(X_te, y_te)
    cv_acc = cross_val_score(model, X_cancer, y_cancer, cv=5).mean()
    results_ensemble[name] = cv_acc
    print(f'{name:<25} {test_acc:>10.4f} {cv_acc:>10.4f}')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
names_e = list(results_ensemble.keys())
scores_e = list(results_ensemble.values())
colors_e = plt.cm.viridis(np.linspace(0.2, 0.9, len(names_e)))
bars = ax.bar(names_e, scores_e, color=colors_e, alpha=0.8)
for bar, score in zip(bars, scores_e):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticklabels(names_e, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Ensemble Methods: 5-Fold CV Accuracy Comparison')
ax.set_ylim(0.9, 1.0)
plt.tight_layout()
plt.show()


## Ringkasan: Key Takeaways Chapter 8

| Method | Parallelizable | Bias | Variance | Best For |
|---|---|---|---|---|
| **Decision Tree** | - | High (shallow) / Low (deep) | Low (shallow) / High (deep) | Interpretable baseline |
| **Bagging** | Yes | Similar to base | Lower | High-variance base learners |
| **Random Forest** | Yes | Low | Low | General purpose, robust |
| **Extra Trees** | Yes | Lower | Even lower | Fast training, large data |
| **AdaBoost** | No | Low | Low | Clean data, binary classification |
| **Gradient Boosting** | No | Very Low | Low-Medium | Best accuracy, structured data |

### Hyperparameter Penting
- **Random Forest**: `n_estimators`, `max_features`, `max_depth`, `min_samples_leaf`
- **Gradient Boosting**: `n_estimators`, `learning_rate`, `max_depth`, `subsample`
- **Kaidah**: semakin besar `n_estimators`, semakin baik; namun computational cost meningkat linear
- **Tuning**: `learning_rate` dan `n_estimators` selalu berinteraksi — gunakan learning rate kecil dengan lebih banyak trees
